# সহজ ভাষায় Notebook Guide

এই notebook **সম্পূর্ণ self-contained** — project folder লাগে না। প্রতিটা concept একটা ছোট FastAPI app বানিয়ে in-process চালিয়ে দেখানো হয়েছে (`TestClient` / `httpx.ASGITransport`, কোনো port খুলতে হয় না)। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন। দরকার: `fastapi`, `httpx`, `ollama`, `pandas`।
2. Section 5-6 real Ollama (`llama3.1:8b`) লাগে: `ollama serve`। বাকিগুলো Ollama ছাড়াই চলে।
3. প্রতিটা experiment-এর আগে অনুমান লিখুন, তারপর output-এর সাথে মেলান।

# Serving LLM Apps with FastAPI — Lecture Notebook

1. Contract: Pydantic schema → OpenAPI
2. Async বনাম blocking — ৪টা request, তিন রকম endpoint
3. `lifespan` — কখন কী চলে
4. ২০২৬-এর দুটো ফাঁদ: strict Content-Type আর 401
5. Streaming: built-in SSE + live llama3.1:8b, TTFT বনাম total
6. Latency anatomy: prompt যত লম্বা, prefill তত ধীর
7. Backpressure: semaphore → 503 + Retry-After
8. Capacity math: Little's law
9. Exact cache + corpus version, safe upload, graceful degradation

In [1]:
import asyncio, json, time, hashlib, io
from contextlib import asynccontextmanager

import fastapi, starlette, httpx, ollama, pandas as pd
from fastapi import FastAPI, HTTPException, Request, Security, UploadFile, File
from fastapi.concurrency import run_in_threadpool
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

MODEL = "llama3.1:8b"
print("fastapi", fastapi.__version__, "| starlette", starlette.__version__, "| httpx", httpx.__version__)
try:
    names = [m.model for m in ollama.Client().list().models]
    print("ollama models:", names, "| has", MODEL, ":", MODEL in names)
except Exception as e:
    print("Ollama not reachable — sections 5-6 will fail:", e)

fastapi 0.162.0 | starlette 1.9.0 | httpx 0.28.1
ollama models: ['llama3.1:8b'] | has llama3.1:8b : True


## 1. Contract: Pydantic Schema → OpenAPI

Client আপনার code দেখে না, দেখে **contract**। Pydantic model-ই validation, documentation আর error message — তিনটাই। `/docs` (Swagger) এই `openapi.json` থেকে আঁকা হয়।

In [2]:
class AskRequest(BaseModel):
    question: str = Field(min_length=1, max_length=2000)     # input-এর সীমা = OWASP LLM10
    top_k: int = Field(default=4, ge=1, le=10)

class AskResponse(BaseModel):
    answer: str
    found: bool                                              # grounding contract-এর অংশ
    citations: list[int]

app1 = FastAPI(title="contract-demo")

@app1.post("/v1/ask", response_model=AskResponse)
async def ask(body: AskRequest):
    return AskResponse(answer=f"echo: {body.question}", found=True, citations=[1])

c = TestClient(app1)
print(c.post("/v1/ask", json={"question": "hi"}).json())
print(c.post("/v1/ask", json={"question": "", "top_k": 50}).status_code, "<- validation error")
schema = c.get("/openapi.json").json()
print("paths:", list(schema["paths"]), "| request fields:", list(schema["components"]["schemas"]["AskRequest"]["properties"]))

{'answer': 'echo: hi', 'found': True, 'citations': [1]}
422 <- validation error
paths: ['/v1/ask'] | request fields: ['question', 'top_k']


## 2. Async বনাম Blocking

`async def` endpoint event loop-এ চলে; plain `def` endpoint threadpool-এ (default ৪০ thread)। `async def`-এর ভেতরে blocking call (sync Ollama client, Chroma, sentence-transformers, pypdf) পুরো loop আটকে দেয়। নিচে `time.sleep(0.5)` সেই blocking library-র ভূমিকায়। **অনুমান করুন:** ৪টা একসাথে request — তিনটা endpoint-এ wall time কত?

In [3]:
app2 = FastAPI()

@app2.get("/async-blocking")
async def async_blocking():
    time.sleep(0.5)                         # BUG: blocks the event loop
    return {"ok": True}

@app2.get("/sync-def")
def sync_def():
    time.sleep(0.5)                         # FastAPI runs plain def in the threadpool
    return {"ok": True}

@app2.get("/async-offloaded")
async def async_offloaded():
    await run_in_threadpool(time.sleep, 0.5)
    return {"ok": True}

async def burst(app, path, n=4):
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://t") as ac:
        t0 = time.perf_counter()
        await asyncio.gather(*(ac.get(path) for _ in range(n)))
        return round(time.perf_counter() - t0, 2)

pd.DataFrame([{"endpoint": p, "wall_s_for_4_requests": await burst(app2, p)}
              for p in ["/async-blocking", "/sync-def", "/async-offloaded"]])

,endpoint,wall_s_for_4_requests
0,/async-blocking,2.01
1,/sync-def,0.51
2,/async-offloaded,0.51


**পড়ার নিয়ম:** প্রথমটা ≈২ s (৪ × ০.৫, একটার পর একটা), বাকি দুটো ≈০.৫ s। Production-এ এর মানে: একজন user-এর PDF parse হওয়ার সময় সবার `/health` পর্যন্ত আটকে থাকে, আর orchestrator ভাবে service মরে গেছে।

## 3. `lifespan` — কখন কী চলে

`@app.on_event("startup")` Starlette 1.0-এ মুছে গেছে। `lifespan`: `yield`-এর আগে startup (model load, client তৈরি, warmup), পরে shutdown। `with TestClient(app)` lifespan চালায়; শুধু `TestClient(app)` না।

In [4]:
events = []

@asynccontextmanager
async def lifespan(app):
    events.append("startup: load embedder, open Chroma, warm the LLM")
    app.state.model = "loaded-once"
    yield
    events.append("shutdown: close clients")

app3 = FastAPI(lifespan=lifespan)

@app3.get("/which")
async def which(request: Request):
    return {"model": request.app.state.model}

with TestClient(app3) as c3:
    events.append(f"request -> {c3.get('/which').json()}")
    events.append(f"request -> {c3.get('/which').json()}")
events

['startup: load embedder, open Chroma, warm the LLM',
 "request -> {'model': 'loaded-once'}",
 "request -> {'model': 'loaded-once'}",
 'shutdown: close clients']

## 4. ২০২৬-এর দুটো ফাঁদ

* **Strict Content-Type (0.132+):** JSON body-র সাথে `Content-Type: application/json` না দিলে **422**। `curl -d '{...}'` header ছাড়া = 422 — student-দের সবচেয়ে সাধারণ প্রথম error।
* **401, 403 না (0.122+):** security class credential না পেলে এখন 401 দেয়।

In [5]:
from fastapi.security import APIKeyHeader

app4 = FastAPI()
key_header = APIKeyHeader(name="X-API-Key")               # auto_error=True -> missing key = 401

@app4.post("/v1/ask")
async def ask4(body: AskRequest, key: str = Security(key_header)):
    return {"ok": True}

c4 = TestClient(app4)
good = c4.post("/v1/ask", json={"question": "q"}, headers={"X-API-Key": "k"})
no_ct = c4.post("/v1/ask", content=json.dumps({"question": "q"}), headers={"X-API-Key": "k", "content-type": "text/plain"})
no_key = c4.post("/v1/ask", json={"question": "q"})
pd.DataFrame([{"case": "JSON + key", "status": good.status_code},
              {"case": "body sent as text/plain", "status": no_ct.status_code},
              {"case": "no API key", "status": no_key.status_code}])

,case,status
0,JSON + key,200
1,body sent as text/plain,422
2,no API key,401


## 5. Streaming: Built-in SSE + Live llama3.1:8b

FastAPI 0.135 থেকে `fastapi.sse` built-in। Docs-এর pattern: `response_class=EventSourceResponse` আর endpoint নিজেই async generator; প্রতিটা `yield`-এ একটা `ServerSentEvent`। আগে citation (retrieval-এর পরেই, prefill চলাকালীন screen-এ কিছু), তারপর token, শেষে `done`।

⚠️ এই pattern-এ generator-এর কোড response শুরুর **পরে** চলে — তখন status 200 commit হয়ে গেছে, 503 দেওয়া যায় না। Project তাই slot আগে নিয়ে তারপর response ফেরত দেয় (context.md §3.4)।

In [6]:
from collections.abc import AsyncIterable
from fastapi.sse import EventSourceResponse, ServerSentEvent

PASSAGE = "Employees get 12 paid sick days per year. Tell your manager before 10am on the first day."
app5 = FastAPI()
llm = ollama.AsyncClient()

@app5.post("/v1/ask/stream", response_class=EventSourceResponse)
async def ask_stream(body: AskRequest, request: Request) -> AsyncIterable[ServerSentEvent]:
    yield ServerSentEvent(data=[{"n": 1, "source": "handbook.pdf", "page": 1}], event="citations")
    stream = await llm.chat(model=MODEL, stream=True, keep_alive="30m",
                            options={"temperature": 0, "seed": 42, "num_predict": 60},
                            messages=[{"role": "system", "content": "Answer ONLY from the context. Cite [1]."},
                                      {"role": "user", "content": f"CONTEXT:\n[1] {PASSAGE}\n\nQUESTION: {body.question}"}])
    async for part in stream:
        if await request.is_disconnected():
            break
        yield ServerSentEvent(data=part.message.content, event="token")
    yield ServerSentEvent(raw_data="[DONE]", event="done")

t0 = time.perf_counter(); ttft = None; names = []
with TestClient(app5) as c5, c5.stream("POST", "/v1/ask/stream", json={"question": "How many sick days do I get?"}) as r:
    headers = {k: r.headers[k] for k in ("content-type", "cache-control", "x-accel-buffering")}
    for line in r.iter_lines():
        if line.startswith("event: "):
            names.append(line[7:])
            if line == "event: token" and ttft is None:
                ttft = time.perf_counter() - t0
total = time.perf_counter() - t0
print(headers)
print(f"events: {names[0]}, {names.count('token')} x token, {names[-1]} | TTFT {ttft:.2f}s | total {total:.2f}s")

{'content-type': 'text/event-stream; charset=utf-8', 'cache-control': 'no-cache', 'x-accel-buffering': 'no'}
events: citations, 5 x token, done | TTFT 3.80s | total 4.01s


**লক্ষ্য করুন:** `Cache-Control: no-cache` আর `X-Accel-Buffering: no` (nginx যেন token জমিয়ে না রাখে) built-in pattern নিজেই দেয়। প্রথম call-এ model cold হলে TTFT-এ load time ঢোকে — আবার চালিয়ে দেখুন।

## 6. Latency Anatomy: Prompt যত লম্বা, Prefill তত ধীর

Ollama প্রতিটা উত্তরে nanosecond-এ timing দেয়: `load_duration` (cold start), `prompt_eval_count/_duration` (**prefill** = prompt পড়া → TTFT), `eval_count/_duration` (**decode** = token লেখা)। নিচে একই প্রশ্ন, context ১, ৪ আর ১২টা passage-এ। প্রতিবার শুরুতে একটা nonce, যাতে Ollama-র prompt-prefix cache কাজে না লাগে (নইলে prefill মিথ্যা রকম ছোট দেখায়)।

In [7]:
client = ollama.Client()
FILLER = ("Remote work is allowed up to 3 days per week with manager approval. Core hours are 11:00 to 15:00. "
          "A one-time home-office allowance of 300 USD covers a desk, chair or monitor. ")
rows = []
for n_passages in (1, 4, 12):
    ctx = "\n".join(f"[{i}] {PASSAGE if i == 1 else FILLER}" for i in range(1, n_passages + 1))
    r = client.chat(model=MODEL, keep_alive="30m", options={"temperature": 0, "seed": 42, "num_predict": 30, "num_ctx": 8192},
                    messages=[{"role": "user", "content": f"(nonce {time.time_ns()})\nCONTEXT:\n{ctx}\n\nHow many sick days? Cite."}])
    rows.append({"passages": n_passages, "prompt_tokens": r.prompt_eval_count,
                 "load_ms": round(r.load_duration / 1e6), "prefill_ms": round(r.prompt_eval_duration / 1e6),
                 "prefill_tok_per_s": round(r.prompt_eval_count / (r.prompt_eval_duration / 1e9)),
                 "decode_tok_per_s": round(r.eval_count / (r.eval_duration / 1e9), 1)})
anatomy = pd.DataFrame(rows); anatomy

,passages,prompt_tokens,load_ms,prefill_ms,prefill_tok_per_s,decode_tok_per_s
0,1,56,3591,415,135,19.3
1,4,206,4,1348,153,21.2
2,12,606,1,3773,161,20.7


**পড়ার নিয়ম:** prompt token বাড়লে prefill_ms প্রায় সমানুপাতে বাড়ে; decode tok/s প্রায় একই থাকে (memory-bandwidth-bound)। আর প্রথম row-এর `load_ms` দেখুন: section 5-এর call Ollama-র default `num_ctx` দিয়েছিল, এখানে `num_ctx=8192` — **option বদলালে Ollama model আবার load করে** (কয়েক সেকেন্ড)। তাই project-এ warmup আর আসল call একই `num_ctx` পাঠায়। Hosted API-র নিয়ম "prompt অর্ধেক করলে latency ১-৫% কমে" laptop-এ খাটে না — এখানে **top-k আর chunk size সরাসরি TTFT**।

## 7. Backpressure: Semaphore → 503 + Retry-After

Ollama default-এ একটাই request চালায় (`OLLAMA_NUM_PARALLEL=1`) আর ৫১২টা পর্যন্ত চুপচাপ queue করে। API-র নিজের gate চাই: slot-এর সমান semaphore, ছোট অপেক্ষা, তারপর দ্রুত 503। নিচে নকল LLM (০.৫ s), ১টা slot, ০.৩ s অপেক্ষা, ৫টা একসাথে request।

In [8]:
app7 = FastAPI()
SLOTS = asyncio.Semaphore(1)

async def fake_llm():
    await asyncio.sleep(0.5)
    return "answer"

@app7.post("/v1/ask")
async def ask7():
    try:
        await asyncio.wait_for(SLOTS.acquire(), timeout=0.3)
    except TimeoutError:
        raise HTTPException(503, "Model busy, retry later", headers={"Retry-After": "1"})
    try:
        return {"answer": await fake_llm()}
    finally:
        SLOTS.release()

async def five():
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app7), base_url="http://t") as ac:
        async def one(i):
            t0 = time.perf_counter(); r = await ac.post("/v1/ask")
            return {"req": i, "status": r.status_code, "seconds": round(time.perf_counter() - t0, 2),
                    "retry_after": r.headers.get("retry-after")}
        return await asyncio.gather(*(one(i) for i in range(5)))

pd.DataFrame(await five())

,req,status,seconds,retry_after
0,0,200,0.5,NaN
1,1,503,0.3,1
2,2,503,0.3,1
3,3,503,0.3,1
4,4,503,0.3,1


একজন সাথে সাথে চলে; বাকিরা ০.৩ s অপেক্ষা করে slot পায় না → 503, ০.৩ s-এ, এক ঘণ্টা পরে না। Client বা load balancer `Retry-After` দেখে পরে আসে বা অন্য replica-য় যায়। **`--workers 8` এটা ভাঙে:** semaphore process-প্রতি, তাই ৮ worker = ৮টা slot একই একটা GPU-র বিরুদ্ধে।

## 8. Capacity Math: Little's Law

L = λ·W (system-এ গড় request = আগমনের হার × গড় সময়)। এক slot-এ সর্বোচ্চ throughput = 1 / W। Section 6-এর মাপ থেকে নিজের মেশিনের হিসাব:

In [9]:
row = anatomy.iloc[1]                                  # 4 passages ~ a typical RAG prompt
answer_tokens = 60
W = row.prefill_ms / 1000 + answer_tokens / row.decode_tok_per_s
per_min = 60 / W
print(f"W = prefill {row.prefill_ms/1000:.2f}s + {answer_tokens} tokens / {row.decode_tok_per_s} tok/s = {W:.2f}s per answer")
print(f"one slot -> at most {per_min:.0f} answers/minute; plan for 60% utilisation -> {0.6 * per_min:.0f}/minute")
for users_per_min in (5, 20, 60):
    print(f"{users_per_min:3} questions/min -> utilisation {users_per_min / per_min:.0%}",
          "OK" if users_per_min / per_min < 0.7 else "-> queue grows: shed load (503/429) or add capacity")

W = prefill 1.35s + 60 tokens / 21.2 tok/s = 4.18s per answer
one slot -> at most 14 answers/minute; plan for 60% utilisation -> 9/minute
  5 questions/min -> utilisation 35% OK
 20 questions/min -> utilisation 139% -> queue grows: shed load (503/429) or add capacity
 60 questions/min -> utilisation 418% -> queue grows: shed load (503/429) or add capacity


## 9. Exact Cache, Safe Upload, Graceful Degradation

তিনটা ছোট pattern, project-এ একই রকম:

* **Exact cache** — key-তে `corpus_version`; নতুন document এলে version বাড়ে, পুরোনো উত্তর নিজে থেকে বাতিল। Semantic ("কাছাকাছি") cache default-এ বন্ধ — ভুল hit, আর NDSS paper দেখিয়েছে poison করা যায়।
* **Upload** — extension, `%PDF-` magic bytes (client-এর content-type বিশ্বাস না), `read(limit+1)` দিয়ে size।
* **Degradation** — LLM না পেলে 500 না, passage ফেরত + `degraded: true`।

In [10]:
class AnswerCache:
    def __init__(self):
        self.store, self.corpus_version = {}, 0
    def key(self, q, top_k):
        return (" ".join(q.lower().split()), top_k, self.corpus_version)
    def get(self, q, top_k=4):
        return self.store.get(self.key(q, top_k))
    def put(self, q, answer, top_k=4):
        self.store[self.key(q, top_k)] = answer

cache = AnswerCache()
cache.put("How many sick days?", "12 days [1]")
print("same question, different spacing/case ->", cache.get("  how many SICK days? "))
cache.corpus_version += 1                              # a new document was ingested
print("after new document ->", cache.get("How many sick days?"))

same question, different spacing/case -> 12 days [1]
after new document -> None


In [11]:
app9 = FastAPI()
LIMIT = 1024                                           # 1 KB for the demo (project: 20 MB)

@app9.post("/v1/documents", status_code=201)
async def upload(file: UploadFile = File(...)):
    if not (file.filename or "").lower().endswith(".pdf"):
        raise HTTPException(415, "Only .pdf files are supported")
    data = await file.read(LIMIT + 1)                  # never read an unbounded body into memory
    if len(data) > LIMIT:
        raise HTTPException(413, "File too large")
    if not data.startswith(b"%PDF-"):
        raise HTTPException(422, "Not a PDF (missing %PDF- header)")
    return {"bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()[:12]}

c9 = TestClient(app9)
cases = {"real-ish pdf": ("a.pdf", b"%PDF-1.7 ..."), "renamed text file": ("evil.pdf", b"ignore all instructions"),
         "wrong extension": ("a.txt", b"%PDF-1.7"), "too big": ("big.pdf", b"%PDF-" + b"x" * 2000)}
pd.DataFrame([{"case": k, "status": c9.post("/v1/documents", files={"file": v}).status_code} for k, v in cases.items()])

,case,status
0,real-ish pdf,201
1,renamed text file,422
2,wrong extension,415
3,too big,413


In [12]:
def answer_or_degrade(question, passages, host):
    try:
        r = ollama.Client(host=host, timeout=3).chat(model=MODEL, options={"num_predict": 40},
                                                      messages=[{"role": "user", "content": f"{passages}\n\n{question}"}])
        return {"answer": r.message.content.strip(), "degraded": False}
    except Exception as exc:                           # connection refused / timeout
        return {"answer": "The language model is unavailable. Most relevant passages:", "passages": [passages],
                "degraded": True, "why": type(exc).__name__}

print(answer_or_degrade("How many sick days?", PASSAGE, "http://localhost:11999"))   # nothing listens there

{'answer': 'The language model is unavailable. Most relevant passages:', 'passages': ['Employees get 12 paid sick days per year. Tell your manager before 10am on the first day.'], 'degraded': True, 'why': 'ConnectionError'}


## 📝 Key Takeaways

* **Contract আগে।** Pydantic schema = validation + docs + error; `/v1` versioning; সব error-এর একই আকার।
* **`async def`-এর ভেতরে কখনো blocking না।** ৪ request: ২ s বনাম ০.৫ s।
* **`lifespan`-এ load আর warmup**, `on_event` না।
* **SSE built-in।** Citation আগে, token পরে, `done` শেষে। Status প্রথম byte-এ commit — 503/401 stream-এর আগে।
* **Ollama একটা slot।** Gate = slot, ছোট অপেক্ষা, দ্রুত 503 + `Retry-After`; বেশি worker সমাধান না।
* **Prefill = prompt-এর দৈর্ঘ্য।** Laptop-এ top-k আর chunk size সরাসরি TTFT; stable অংশ আগে রাখলে prefix cache।
* **Exact cache + corpus version; upload-এ magic bytes আর size; LLM down হলে degrade।**

## 🧠 Brain Teasers

1. Section 2-এ `n=40` আর `n=41` দিয়ে `/sync-def` চালান। Wall time হঠাৎ লাফায় কেন? (hint: threadpool-এর ৪০ token)
2. Section 5-এর endpoint-এ `yield`-এর আগে `raise HTTPException(503)` বসান। Client কোন status পায়? কেন?
3. Section 6-এ nonce সরিয়ে দুবার চালান। দ্বিতীয়বার prefill_ms কত? Benchmark-এ এর মানে কী?
4. Section 7-এ `timeout=0.3`-কে `5` করুন। কয়টা 503 হয়, আর শেষ request কতক্ষণ অপেক্ষা করে? কোনটা user-এর জন্য ভালো?
5. Section 9-এর upload-এ এমন একটা file বানান যেটা `%PDF-` দিয়ে শুরু কিন্তু PDF না। কোন layer সেটা ধরবে?